In [2]:
# ============================================================
# Install:
%pip install -U langgraph typing_extensions
# ============================================================

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 23.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.16
    Uninstalling langchain-protocol-0.0.16:
      Successfully uninstalled langchain-protocol-0.0.16
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.4
    Uninstalling langgraph-1.2.4:
      Successfully uninstalled langgraph-1.2.4


In [3]:

from typing import Dict, List, Any, Literal, Optional
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
import statistics
import copy


# ============================================================
# 1. Manual KPI + Target Configuration
#    Initially we manually define objective -> KPIs -> targets.
#    Later these targets are improved using historical data.
# ============================================================

OBJECTIVE_KPI_CONFIG = {
    "bookings": {
        "primary_kpi": "ROAS",
        "secondary_kpis": ["CPA", "CTR", "Conversion Rate", "Revenue", "Conversions"],
        "manual_targets": {
            "target_roas": 4.0,
            "max_cpa_pct_of_budget": 0.01,
            "target_ctr": 4.0,
            "target_conversion_rate": 3.0
        }
    },

    "revenue": {
        "primary_kpi": "ROAS",
        "secondary_kpis": ["CPA", "CTR", "Conversion Rate", "Revenue"],
        "manual_targets": {
            "target_roas": 4.5,
            "max_cpa_pct_of_budget": 0.01,
            "target_ctr": 4.0,
            "target_conversion_rate": 3.0
        }
    },

    "sales": {
        "primary_kpi": "ROAS",
        "secondary_kpis": ["CPA", "CTR", "Conversion Rate", "Revenue"],
        "manual_targets": {
            "target_roas": 4.0,
            "max_cpa_pct_of_budget": 0.01,
            "target_ctr": 4.0,
            "target_conversion_rate": 3.0
        }
    },

    "signups": {
        "primary_kpi": "CPA",
        "secondary_kpis": ["Conversions", "CTR", "CPC", "Conversion Rate"],
        "manual_targets": {
            "target_roas": None,
            "max_cpa_pct_of_budget": 0.005,
            "target_ctr": 3.0,
            "target_conversion_rate": 4.0
        }
    },

    "leads": {
        "primary_kpi": "CPA",
        "secondary_kpis": ["Conversions", "CTR", "CPC", "Conversion Rate"],
        "manual_targets": {
            "target_roas": None,
            "max_cpa_pct_of_budget": 0.006,
            "target_ctr": 3.0,
            "target_conversion_rate": 4.0
        }
    },

    "trials": {
        "primary_kpi": "CPA",
        "secondary_kpis": ["Trial Starts", "CTR", "CPC", "Conversion Rate"],
        "manual_targets": {
            "target_roas": None,
            "max_cpa_pct_of_budget": 0.006,
            "target_ctr": 3.5,
            "target_conversion_rate": 4.0
        }
    },

    "awareness": {
        "primary_kpi": "CPM",
        "secondary_kpis": ["Impressions", "Reach", "CTR", "Frequency"],
        "manual_targets": {
            "target_roas": None,
            "max_cpa_pct_of_budget": None,
            "target_ctr": 2.0,
            "target_conversion_rate": None,
            "target_cpm": 150
        }
    },

    "reach": {
        "primary_kpi": "Reach",
        "secondary_kpis": ["Impressions", "CPM", "Frequency", "CTR"],
        "manual_targets": {
            "target_roas": None,
            "max_cpa_pct_of_budget": None,
            "target_ctr": 2.0,
            "target_conversion_rate": None,
            "target_cpm": 150
        }
    }
}


DEFAULT_OBJECTIVE_CONFIG = {
    "primary_kpi": "CPA",
    "secondary_kpis": ["ROAS", "CTR", "Conversion Rate", "Conversions"],
    "manual_targets": {
        "target_roas": 3.0,
        "max_cpa_pct_of_budget": 0.008,
        "target_ctr": 3.0,
        "target_conversion_rate": 2.5
    }
}


# ============================================================
# 2. Benchmark DB
#    Initially static benchmark values.
#    Later replace this with real benchmark table/API.
# ============================================================

BENCHMARK_DB = {
    "Google Ads": {
        "cpa_range": [250, 900],
        "roas_range": [2.5, 6.0],
        "ctr_range": [2.0, 7.0],
        "conversion_rate_range": [1.5, 6.0],
        "cpm_range": [80, 300]
    },

    "Meta": {
        "cpa_range": [200, 800],
        "roas_range": [2.0, 5.0],
        "ctr_range": [1.0, 5.0],
        "conversion_rate_range": [1.0, 5.0],
        "cpm_range": [70, 280]
    },

    "Email": {
        "cpa_range": [20, 250],
        "roas_range": [5.0, 20.0],
        "ctr_range": [3.0, 15.0],
        "conversion_rate_range": [2.0, 12.0],
        "cpm_range": [10, 80]
    }
}


DEFAULT_BENCHMARK = {
    "cpa_range": [300, 1000],
    "roas_range": [1.5, 4.0],
    "ctr_range": [1.0, 4.0],
    "conversion_rate_range": [1.0, 4.0],
    "cpm_range": [80, 300]
}


# ============================================================
# 3. Simple Historical Memory Layer
#    For now this is in-memory sample data.
#    Later replace this with PostgreSQL queries.
# ============================================================

class MetricsMemoryStore:
    """
    This simulates historical campaign memory.

    In production:
    - replace this class with PostgreSQL repository
    - store campaign_id, objective, product, channel, CPA, ROAS, CTR, CVR, spend, conversions
    """

    def __init__(self):
        self.history = [
            {
                "product": "Flight Sale",
                "objective": "bookings",
                "channel": "Google Ads",
                "cpa": 420,
                "roas": 4.8,
                "ctr": 5.2,
                "conversion_rate": 3.4,
                "spend": 50000,
                "conversions": 120,
                "status": "winner"
            },
            {
                "product": "Flight Sale",
                "objective": "bookings",
                "channel": "Meta",
                "cpa": 620,
                "roas": 3.1,
                "ctr": 2.8,
                "conversion_rate": 2.2,
                "spend": 30000,
                "conversions": 48,
                "status": "learning"
            },
            {
                "product": "Flight Sale",
                "objective": "bookings",
                "channel": "Email",
                "cpa": 90,
                "roas": 9.5,
                "ctr": 8.5,
                "conversion_rate": 6.0,
                "spend": 10000,
                "conversions": 111,
                "status": "winner"
            }
        ]

    def get_relevant_history(
        self,
        product: str,
        objective: str,
        channels: List[str]
    ) -> List[Dict[str, Any]]:
        product_lower = product.lower()
        objective_lower = objective.lower()

        return [
            row for row in self.history
            if row["objective"].lower() == objective_lower
            and row["channel"] in channels
            and product_lower in row["product"].lower()
        ]

    def save_campaign_feedback(self, feedback: Dict[str, Any]) -> None:
        """
        This method will be called after live campaign performance comes back.
        For now it appends to in-memory list.
        In production it should insert into PostgreSQL.
        """
        self.history.append(feedback)


memory_store = MetricsMemoryStore()


# ============================================================
# 4. LangGraph State
# ============================================================

class MetricsState(TypedDict, total=False):
    # Input
    campaign_id: str
    product: str
    objective: str
    total_budget: float

    # Output from GTM Agent
    gtm_plan: Dict[str, Any]

    # Manual KPI config
    objective_config: Dict[str, Any]
    primary_kpi: str
    secondary_kpis: List[str]

    # Target setting
    manual_targets: Dict[str, Any]
    historical_performance: List[Dict[str, Any]]
    learned_targets: Dict[str, Any]
    global_targets: Dict[str, Any]
    benchmark_data: Dict[str, Any]
    channel_targets: Dict[str, Any]

    # Alignment and negotiation
    issues: List[Dict[str, Any]]
    negotiation_round: int
    final_status: Literal["aligned", "warnings", "halt"]
    final_message: str

    # Optional Sprint 6 feedback-loop fields
    live_performance: Dict[str, Any]
    optimization_recommendations: List[Dict[str, Any]]
    channel_ranking: List[Dict[str, Any]]


# ============================================================
# 5. Utility Functions
# ============================================================

def normalize_objective(objective: str) -> str:
    return objective.strip().lower()


def safe_median(values: List[float]) -> Optional[float]:
    values = [v for v in values if v is not None]
    if not values:
        return None
    return statistics.median(values)


def clamp(value: Optional[float], low: float, high: float) -> Optional[float]:
    if value is None:
        return None
    return max(low, min(value, high))


def round_or_none(value: Optional[float], digits: int = 2) -> Optional[float]:
    if value is None:
        return None
    return round(value, digits)


# ============================================================
# 6. KPI Selector Agent
#    Manually maps objective -> primary and secondary KPIs.
# ============================================================

def kpi_selector_agent(state: MetricsState) -> Dict[str, Any]:
    objective = normalize_objective(state["objective"])

    objective_config = OBJECTIVE_KPI_CONFIG.get(
        objective,
        DEFAULT_OBJECTIVE_CONFIG
    )

    return {
        "objective_config": objective_config,
        "primary_kpi": objective_config["primary_kpi"],
        "secondary_kpis": objective_config["secondary_kpis"]
    }


# ============================================================
# 7. Manual Target Setter Agent
#    Sets initial target values from manual configuration.
# ============================================================

def manual_target_setter_agent(state: MetricsState) -> Dict[str, Any]:
    budget = state["total_budget"]
    objective_config = state["objective_config"]
    raw_manual_targets = objective_config["manual_targets"]

    max_cpa_pct = raw_manual_targets.get("max_cpa_pct_of_budget")

    if max_cpa_pct is not None:
        max_cpa = budget * max_cpa_pct
    else:
        max_cpa = None

    manual_targets = {
        "target_roas": raw_manual_targets.get("target_roas"),
        "max_cpa": max_cpa,
        "target_ctr": raw_manual_targets.get("target_ctr"),
        "target_conversion_rate": raw_manual_targets.get("target_conversion_rate"),
        "target_cpm": raw_manual_targets.get("target_cpm")
    }

    return {
        "manual_targets": manual_targets
    }


# ============================================================
# 8. Historical Data Loader Agent
#    Loads previous campaign data.
#    This is where PostgreSQL will be connected later.
# ============================================================

def historical_data_loader_agent(state: MetricsState) -> Dict[str, Any]:
    product = state.get("product", "")
    objective = normalize_objective(state["objective"])
    selected_channels = state["gtm_plan"]["recommended_channels"]

    historical_performance = memory_store.get_relevant_history(
        product=product,
        objective=objective,
        channels=selected_channels
    )

    return {
        "historical_performance": historical_performance
    }


# ============================================================
# 9. Learning Target Agent
#    Learns updated targets from previous performance.
# ============================================================

def learning_target_agent(state: MetricsState) -> Dict[str, Any]:
    history = state.get("historical_performance", [])
    manual_targets = state["manual_targets"]

    if not history:
        return {
            "learned_targets": {},
            "global_targets": manual_targets
        }

    roas_values = [row.get("roas") for row in history if row.get("roas") is not None]
    cpa_values = [row.get("cpa") for row in history if row.get("cpa") is not None]
    ctr_values = [row.get("ctr") for row in history if row.get("ctr") is not None]
    cvr_values = [
        row.get("conversion_rate")
        for row in history
        if row.get("conversion_rate") is not None
    ]

    median_roas = safe_median(roas_values)
    median_cpa = safe_median(cpa_values)
    median_ctr = safe_median(ctr_values)
    median_cvr = safe_median(cvr_values)

    learned_targets = {}

    # For ROAS:
    # If past ROAS exists, set target slightly above median performance.
    # But do not make it unrealistically aggressive.
    if median_roas is not None:
        learned_targets["target_roas"] = round(median_roas * 1.10, 2)

    # For CPA:
    # If past CPA exists, target slightly lower than median CPA.
    # This means system tries to improve efficiency.
    if median_cpa is not None:
        learned_targets["max_cpa"] = round(median_cpa * 0.95, 2)

    # For CTR:
    # Target slightly above historical median CTR.
    if median_ctr is not None:
        learned_targets["target_ctr"] = round(median_ctr * 1.05, 2)

    # For Conversion Rate:
    # Target slightly above historical median CVR.
    if median_cvr is not None:
        learned_targets["target_conversion_rate"] = round(median_cvr * 1.05, 2)

    # Merge manual + learned.
    # Learned targets override manual only when data exists.
    global_targets = copy.deepcopy(manual_targets)

    for key, value in learned_targets.items():
        if value is not None:
            global_targets[key] = value

    return {
        "learned_targets": learned_targets,
        "global_targets": global_targets
    }


# ============================================================
# 10. Benchmark Checker Agent
#     Fetches benchmark ranges per selected channel.
# ============================================================

def benchmark_checker_agent(state: MetricsState) -> Dict[str, Any]:
    selected_channels = state["gtm_plan"]["recommended_channels"]

    benchmark_data = {}

    for channel in selected_channels:
        benchmark_data[channel] = BENCHMARK_DB.get(
            channel,
            DEFAULT_BENCHMARK
        )

    return {
        "benchmark_data": benchmark_data
    }


# ============================================================
# 11. Channel Target Agent
#     Creates per-channel targets using:
#     manual target + learned target + benchmark sanity check.
# ============================================================

def channel_target_agent(state: MetricsState) -> Dict[str, Any]:
    global_targets = state["global_targets"]
    benchmark_data = state["benchmark_data"]
    history = state.get("historical_performance", [])

    channel_targets = {}

    for channel, benchmark in benchmark_data.items():
        cpa_low, cpa_high = benchmark["cpa_range"]
        roas_low, roas_high = benchmark["roas_range"]
        ctr_low, ctr_high = benchmark["ctr_range"]
        cvr_low, cvr_high = benchmark["conversion_rate_range"]
        cpm_low, cpm_high = benchmark["cpm_range"]

        channel_history = [
            row for row in history
            if row.get("channel") == channel
        ]

        channel_median_roas = safe_median(
            [row.get("roas") for row in channel_history]
        )
        channel_median_cpa = safe_median(
            [row.get("cpa") for row in channel_history]
        )
        channel_median_ctr = safe_median(
            [row.get("ctr") for row in channel_history]
        )
        channel_median_cvr = safe_median(
            [row.get("conversion_rate") for row in channel_history]
        )

        # Start from global targets
        target_cpa = global_targets.get("max_cpa")
        target_roas = global_targets.get("target_roas")
        target_ctr = global_targets.get("target_ctr")
        target_cvr = global_targets.get("target_conversion_rate")
        target_cpm = global_targets.get("target_cpm")

        # If channel-specific historical data exists, improve per channel.
        if channel_median_roas is not None:
            target_roas = channel_median_roas * 1.10

        if channel_median_cpa is not None:
            target_cpa = channel_median_cpa * 0.95

        if channel_median_ctr is not None:
            target_ctr = channel_median_ctr * 1.05

        if channel_median_cvr is not None:
            target_cvr = channel_median_cvr * 1.05

        # Benchmark sanity check
        final_cpa = clamp(target_cpa, cpa_low, cpa_high)
        final_roas = clamp(target_roas, roas_low, roas_high)
        final_ctr = clamp(target_ctr, ctr_low, ctr_high)
        final_cvr = clamp(target_cvr, cvr_low, cvr_high)
        final_cpm = clamp(target_cpm, cpm_low, cpm_high)

        channel_targets[channel] = {
            "target_cpa": round_or_none(final_cpa),
            "target_roas": round_or_none(final_roas),
            "target_ctr": round_or_none(final_ctr),
            "target_conversion_rate": round_or_none(final_cvr),
            "target_cpm": round_or_none(final_cpm),

            "baseline_cpa_range": benchmark["cpa_range"],
            "baseline_roas_range": benchmark["roas_range"],
            "baseline_ctr_range": benchmark["ctr_range"],
            "baseline_conversion_rate_range": benchmark["conversion_rate_range"],
            "baseline_cpm_range": benchmark["cpm_range"],

            "used_historical_data": len(channel_history) > 0
        }

    return {
        "channel_targets": channel_targets
    }


# ============================================================
# 12. GTM Alignment Agent
#     Checks whether metrics targets make sense with GTM plan.
# ============================================================

def gtm_alignment_agent(state: MetricsState) -> Dict[str, Any]:
    issues = []

    gtm_plan = state["gtm_plan"]
    channel_targets = state["channel_targets"]

    channel_priority = gtm_plan.get("channel_priority", {})

    for channel, targets in channel_targets.items():
        target_cpa = targets.get("target_cpa")
        target_roas = targets.get("target_roas")
        target_ctr = targets.get("target_ctr")
        target_cvr = targets.get("target_conversion_rate")

        cpa_low, cpa_high = targets["baseline_cpa_range"]
        roas_low, roas_high = targets["baseline_roas_range"]
        ctr_low, ctr_high = targets["baseline_ctr_range"]
        cvr_low, cvr_high = targets["baseline_conversion_rate_range"]

        # CPA target too aggressive
        if target_cpa is not None and target_cpa < cpa_low:
            issues.append({
                "channel": channel,
                "issue": "CPA target too aggressive",
                "severity": "high",
                "suggestion": f"Increase CPA target to at least {cpa_low}"
            })

        # CPA target too loose
        if target_cpa is not None and target_cpa > cpa_high:
            issues.append({
                "channel": channel,
                "issue": "CPA target too loose",
                "severity": "medium",
                "suggestion": f"Reduce CPA target closer to benchmark max {cpa_high}"
            })

        # ROAS target too aggressive
        if target_roas is not None and target_roas > roas_high:
            issues.append({
                "channel": channel,
                "issue": "ROAS target too aggressive",
                "severity": "high",
                "suggestion": f"Reduce ROAS target to max {roas_high}"
            })

        # ROAS target too weak
        if target_roas is not None and target_roas < roas_low:
            issues.append({
                "channel": channel,
                "issue": "ROAS target too weak",
                "severity": "medium",
                "suggestion": f"Increase ROAS target to at least {roas_low}"
            })

        # CTR target too aggressive
        if target_ctr is not None and target_ctr > ctr_high:
            issues.append({
                "channel": channel,
                "issue": "CTR target too aggressive",
                "severity": "medium",
                "suggestion": f"Reduce CTR target to max {ctr_high}"
            })

        # Conversion rate target too aggressive
        if target_cvr is not None and target_cvr > cvr_high:
            issues.append({
                "channel": channel,
                "issue": "Conversion rate target too aggressive",
                "severity": "medium",
                "suggestion": f"Reduce conversion rate target to max {cvr_high}"
            })

        # GTM priority mismatch
        if channel_priority.get(channel) == "low" and target_roas is not None:
            if target_roas >= roas_high * 0.95:
                issues.append({
                    "channel": channel,
                    "issue": "Low priority channel has high performance expectation",
                    "severity": "medium",
                    "suggestion": "Reduce expectation or change GTM priority"
                })

    return {
        "issues": issues
    }


# ============================================================
# 13. Router
# ============================================================

def metrics_router(state: MetricsState) -> str:
    issues = state.get("issues", [])
    round_no = state.get("negotiation_round", 0)

    if len(issues) == 0:
        return "final"

    if round_no < 3:
        return "negotiate"

    return "final"


# ============================================================
# 14. Negotiation Agent
#     Adjusts targets based on detected issues.
# ============================================================

def negotiation_agent(state: MetricsState) -> Dict[str, Any]:
    round_no = state.get("negotiation_round", 0) + 1

    channel_targets = copy.deepcopy(state["channel_targets"])
    issues = state["issues"]

    for issue in issues:
        channel = issue["channel"]

        if channel not in channel_targets:
            continue

        target = channel_targets[channel]

        cpa_low, cpa_high = target["baseline_cpa_range"]
        roas_low, roas_high = target["baseline_roas_range"]
        ctr_low, ctr_high = target["baseline_ctr_range"]
        cvr_low, cvr_high = target["baseline_conversion_rate_range"]

        if issue["issue"] == "CPA target too aggressive":
            target["target_cpa"] = cpa_low

        elif issue["issue"] == "CPA target too loose":
            target["target_cpa"] = cpa_high

        elif issue["issue"] == "ROAS target too aggressive":
            target["target_roas"] = roas_high

        elif issue["issue"] == "ROAS target too weak":
            target["target_roas"] = roas_low

        elif issue["issue"] == "CTR target too aggressive":
            target["target_ctr"] = ctr_high

        elif issue["issue"] == "Conversion rate target too aggressive":
            target["target_conversion_rate"] = cvr_high

        elif issue["issue"] == "Low priority channel has high performance expectation":
            current_roas = target.get("target_roas")
            if current_roas is not None:
                target["target_roas"] = round(current_roas * 0.90, 2)

    return {
        "negotiation_round": round_no,
        "channel_targets": channel_targets
    }


# ============================================================
# 15. Final Decision Agent
# ============================================================

def final_decision_agent(state: MetricsState) -> Dict[str, Any]:
    issues = state.get("issues", [])

    if len(issues) == 0:
        return {
            "final_status": "aligned",
            "final_message": "Metrics targets are aligned with GTM plan."
        }

    high_issues = [
        issue for issue in issues
        if issue["severity"] == "high"
    ]

    if len(high_issues) > 0:
        return {
            "final_status": "halt",
            "final_message": (
                "Strategy should halt. Some targets are not realistic "
                "for selected channels."
            )
        }

    return {
        "final_status": "warnings",
        "final_message": "Strategy can proceed with warnings."
    }


# ============================================================
# 16. Optional Sprint 6 Agent:
#     Analyze live performance vs targets.
#     This is used after campaign is live.
# ============================================================

def performance_feedback_agent(state: MetricsState) -> Dict[str, Any]:
    """
    This agent is for later optimization phase.

    live_performance expected format:
    {
        "Google Ads": {
            "spend": 20000,
            "cpa": 430,
            "roas": 4.9,
            "ctr": 5.1,
            "conversion_rate": 3.5
        },
        ...
    }
    """

    live_performance = state.get("live_performance", {})
    channel_targets = state.get("channel_targets", {})

    recommendations = []
    ranking = []

    for channel, performance in live_performance.items():
        targets = channel_targets.get(channel, {})

        actual_cpa = performance.get("cpa")
        actual_roas = performance.get("roas")
        actual_ctr = performance.get("ctr")
        actual_cvr = performance.get("conversion_rate")

        target_cpa = targets.get("target_cpa")
        target_roas = targets.get("target_roas")

        score = 0
        status = "learning"

        if actual_roas is not None and target_roas is not None:
            if actual_roas >= target_roas:
                score += 2
            else:
                score -= 2

        if actual_cpa is not None and target_cpa is not None:
            if actual_cpa <= target_cpa:
                score += 2
            else:
                score -= 2

        if actual_ctr is not None and targets.get("target_ctr") is not None:
            if actual_ctr >= targets["target_ctr"]:
                score += 1

        if actual_cvr is not None and targets.get("target_conversion_rate") is not None:
            if actual_cvr >= targets["target_conversion_rate"]:
                score += 1

        if score >= 3:
            status = "winner"
            action = "scale_or_shift_more_budget"
        elif score <= -2:
            status = "loser"
            action = "pause_or_reduce_budget"
        else:
            status = "learning"
            action = "continue_monitoring"

        ranking.append({
            "channel": channel,
            "score": score,
            "status": status
        })

        recommendations.append({
            "channel": channel,
            "status": status,
            "recommended_action": action,
            "reason": {
                "actual_cpa": actual_cpa,
                "target_cpa": target_cpa,
                "actual_roas": actual_roas,
                "target_roas": target_roas
            }
        })

    ranking = sorted(
        ranking,
        key=lambda x: x["score"],
        reverse=True
    )

    return {
        "optimization_recommendations": recommendations,
        "channel_ranking": ranking
    }


# ============================================================
# 17. Function to save feedback into memory
#     Later replace this with PostgreSQL insert.
# ============================================================

def save_live_feedback_to_memory(
    product: str,
    objective: str,
    live_performance: Dict[str, Any]
) -> None:
    for channel, metrics in live_performance.items():
        feedback_row = {
            "product": product,
            "objective": objective,
            "channel": channel,
            "cpa": metrics.get("cpa"),
            "roas": metrics.get("roas"),
            "ctr": metrics.get("ctr"),
            "conversion_rate": metrics.get("conversion_rate"),
            "spend": metrics.get("spend"),
            "conversions": metrics.get("conversions"),
            "status": metrics.get("status", "unknown")
        }

        memory_store.save_campaign_feedback(feedback_row)


# ============================================================
# 18. Build Metrics Strategy Graph
#     This graph is used in Sprint 2 strategy phase.
# ============================================================

def build_metrics_agent():
    graph = StateGraph(MetricsState)

    graph.add_node("kpi_selector", kpi_selector_agent)
    graph.add_node("manual_target_setter", manual_target_setter_agent)
    graph.add_node("historical_data_loader", historical_data_loader_agent)
    graph.add_node("learning_target", learning_target_agent)
    graph.add_node("benchmark_checker", benchmark_checker_agent)
    graph.add_node("channel_target", channel_target_agent)
    graph.add_node("gtm_alignment", gtm_alignment_agent)
    graph.add_node("negotiation", negotiation_agent)
    graph.add_node("final_decision", final_decision_agent)

    graph.add_edge(START, "kpi_selector")
    graph.add_edge("kpi_selector", "manual_target_setter")
    graph.add_edge("manual_target_setter", "historical_data_loader")
    graph.add_edge("historical_data_loader", "learning_target")
    graph.add_edge("learning_target", "benchmark_checker")
    graph.add_edge("benchmark_checker", "channel_target")
    graph.add_edge("channel_target", "gtm_alignment")

    graph.add_conditional_edges(
        "gtm_alignment",
        metrics_router,
        {
            "negotiate": "negotiation",
            "final": "final_decision"
        }
    )

    graph.add_edge("negotiation", "gtm_alignment")
    graph.add_edge("final_decision", END)

    return graph.compile()


# ============================================================
# 19. Build Optional Performance Feedback Graph
#     This is useful in Sprint 6 optimization phase.
# ============================================================

def build_performance_feedback_graph():
    graph = StateGraph(MetricsState)

    graph.add_node("performance_feedback", performance_feedback_agent)

    graph.add_edge(START, "performance_feedback")
    graph.add_edge("performance_feedback", END)

    return graph.compile()


# ============================================================
# 20. Test Metrics Agent
# ============================================================

if __name__ == "__main__":

    # -------------------------
    # Strategy-phase run
    # -------------------------
    metrics_agent = build_metrics_agent()

    input_state = {
        "campaign_id": "CMP_001",
        "product": "Flight Sale",
        "objective": "bookings",
        "total_budget": 100000,

        "gtm_plan": {
            "recommended_channels": ["Google Ads", "Meta", "Email"],
            "channel_priority": {
                "Google Ads": "high",
                "Meta": "medium",
                "Email": "low"
            }
        },

        "negotiation_round": 0
    }

    result = metrics_agent.invoke(input_state)

    print("\n==============================")
    print("METRICS AGENT RESULT")
    print("==============================")

    print("\nPrimary KPI:")
    print(result["primary_kpi"])

    print("\nSecondary KPIs:")
    print(result["secondary_kpis"])

    print("\nManual Targets:")
    print(result["manual_targets"])

    print("\nHistorical Performance Used:")
    print(result["historical_performance"])

    print("\nLearned Targets:")
    print(result["learned_targets"])

    print("\nFinal Global Targets:")
    print(result["global_targets"])

    print("\nBenchmark Data:")
    print(result["benchmark_data"])

    print("\nChannel Targets:")
    print(result["channel_targets"])

    print("\nIssues:")
    print(result["issues"])

    print("\nFinal Status:")
    print(result["final_status"])

    print("\nFinal Message:")
    print(result["final_message"])

    # -------------------------
    # Later: live feedback comes
    # -------------------------
    live_performance = {
        "Google Ads": {
            "spend": 25000,
            "cpa": 390,
            "roas": 5.2,
            "ctr": 5.5,
            "conversion_rate": 3.7,
            "conversions": 64,
            "status": "winner"
        },
        "Meta": {
            "spend": 18000,
            "cpa": 760,
            "roas": 2.4,
            "ctr": 2.1,
            "conversion_rate": 1.8,
            "conversions": 24,
            "status": "loser"
        },
        "Email": {
            "spend": 8000,
            "cpa": 85,
            "roas": 10.5,
            "ctr": 9.0,
            "conversion_rate": 6.5,
            "conversions": 94,
            "status": "winner"
        }
    }

    # Save this feedback so next run learns from it
    save_live_feedback_to_memory(
        product="Flight Sale",
        objective="bookings",
        live_performance=live_performance
    )

    # Optional optimization analysis
    feedback_graph = build_performance_feedback_graph()

    feedback_result = feedback_graph.invoke({
        "channel_targets": result["channel_targets"],
        "live_performance": live_performance
    })

    print("\n==============================")
    print("PERFORMANCE FEEDBACK RESULT")
    print("==============================")

    print("\nOptimization Recommendations:")
    print(feedback_result["optimization_recommendations"])

    print("\nChannel Ranking:")
    print(feedback_result["channel_ranking"])


METRICS AGENT RESULT

Primary KPI:
ROAS

Secondary KPIs:
['CPA', 'CTR', 'Conversion Rate', 'Revenue', 'Conversions']

Manual Targets:
{'target_roas': 4.0, 'max_cpa': 1000.0, 'target_ctr': 4.0, 'target_conversion_rate': 3.0, 'target_cpm': None}

Historical Performance Used:
[{'product': 'Flight Sale', 'objective': 'bookings', 'channel': 'Google Ads', 'cpa': 420, 'roas': 4.8, 'ctr': 5.2, 'conversion_rate': 3.4, 'spend': 50000, 'conversions': 120, 'status': 'winner'}, {'product': 'Flight Sale', 'objective': 'bookings', 'channel': 'Meta', 'cpa': 620, 'roas': 3.1, 'ctr': 2.8, 'conversion_rate': 2.2, 'spend': 30000, 'conversions': 48, 'status': 'learning'}, {'product': 'Flight Sale', 'objective': 'bookings', 'channel': 'Email', 'cpa': 90, 'roas': 9.5, 'ctr': 8.5, 'conversion_rate': 6.0, 'spend': 10000, 'conversions': 111, 'status': 'winner'}]

Learned Targets:
{'target_roas': 5.28, 'max_cpa': 399.0, 'target_ctr': 5.46, 'target_conversion_rate': 3.57}

Final Global Targets:
{'target_roas': 5